In [ ]:
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import interpolate

In [ ]:
desorption_data =[
    (0.997, 712.285),
    (0.973, 710.839),
    (0.935, 708.102),
    (0.909, 706.054),
    (0.900, 705.062),
    (0.881, 662.486),
    (0.856, 411.324),
    (0.829, 288.826),
    (0.806, 238.821),
    (0.750, 175.429),
    (0.706, 152.333),
    (0.656, 135.776),
    (0.606, 124.146),
    (0.553, 114.455),
    (0.502, 106.818),
    (0.451, 100.181),
    (0.402, 94.600),
    (0.350, 88.974),
    (0.300, 84.040),
    (0.253, 79.431),
    (0.201, 74.350),
    (0.141, 68.106)
]

# Массив относительного давления (X)
data_PP0_inp = np.array([point[0] for point in desorption_data])

# Массив объема (Y)
data_V_inp = np.array([point[1] for point in desorption_data])


In [ ]:
#ДАННЫЕ МОГУТ БЫТЬ ЗАДАНЫ ДРУГИМ СПОСОБОМ

#data_PP0_inp = np.array([0.97, 0.99, 0.71, 0.95, 0.90, 0.85, 0.80, 0.77, 0.75, 0.73, 0.98])
#data_V_inp = np.array([161.0, 161.7, 140.9, 160.2, 158.8, 157.2, 155.2, 152.9, 148.9, 145.1, 161.5])

#data = pd.read_csv("File_path", sep=',')
#data_PP0_inp = data['PP0_des'].to_numpy()
#data_V_inp = data['V_des'].to_numpy()

In [ ]:
data_idx_sort = np.argsort(data_PP0_inp)[::-1]
data_PP0 = data_PP0_inp[data_idx_sort]
data_V = data_V_inp[data_idx_sort]
# r_k в Å (эмпирическая формула для N2 при 77 K)
r_k = -4.15 / np.log10(data_PP0)
# t в Å (толщина монослоя 3.54 Å)
t = 3.54 * (-5/np.log(data_PP0))**(1/3)
# r_p в Å
r_p = r_k + t
r_k_mean = []
r_p_mean = []
delta_t = []
delta_V = []
for i in range(len(data_PP0)-1):
    delta_t.append(t[i]-t[i+1])
    delta_V.append(data_V[i]-data_V[i+1])
    r_k_mean.append(np.mean([r_k[i], r_k[i+1]]))
    r_p_mean.append(np.mean([r_p[i], r_p[i+1]]))
delta_t = np.array(delta_t)
delta_V = np.array(delta_V)
r_k_mean = np.array(r_k_mean)
r_p_mean = np.array(r_p_mean)
delta_V_f = 0.0015468*delta_V
s_t = []
V_p = []
S_p = []
sS_p = []
sV_p = []
sum_S = 0.0
sum_V = 0.0
for i in range(len(data_PP0)-1):
    s_t.append(sum_S*delta_t[i]/10000)
    V_p.append((delta_V_f[i]-s_t[i])*(r_p_mean[i]/r_k_mean[i])**2)
    S_p.append(2*V_p[i]/r_p_mean[i]*10**4)
    sum_S += S_p[i]
    sS_p.append(sum_S)
    sum_V += V_p[i]
    sV_p.append(sum_V)

In [ ]:
#d в Å (диаметр пор)

#Работаем с площадью
#Получаем точки из данных
d = 2*r_p_mean
y_df_pr = [0]
for i in range(len(S_p)-1):
    y_df_pr.append(S_p[i+1]/(d[i]-d[i+1]))
y_df_pr_log = [0]
for i in range(len(S_p)-1):
    y_df_pr_log.append(S_p[i+1]/((np.log10(d[i])-np.log10(d[i+1]))))

#Искусственное увеличение точек с помощью аппроксимации функции распределения
sS_p = np.array(sS_p)
idx_sort = np.argsort(d)
sort_d_mean = d[idx_sort]
sort_sS_p = sS_p[idx_sort]
f = interpolate.Akima1DInterpolator(sort_d_mean, sort_sS_p)
x = np.linspace(sort_d_mean[0], sort_d_mean[-1], 500)
df = f.derivative()
y_df = -df(x)
y_df = np.clip(y_df, 0, max(y_df_pr))
y_df_log = y_df * x * np.log(10)

#Интерполяции точек данных
y_df_pr = np.array(y_df_pr)
spline = interpolate.Akima1DInterpolator(sort_d_mean, y_df_pr[idx_sort])
y_spline = spline(x)
y_df_pr_log = np.array(y_df_pr_log)
spline_log = interpolate.Akima1DInterpolator(sort_d_mean, y_df_pr_log[idx_sort])
y_spline_log = spline_log(x)

In [ ]:
#Работаем с объемом
#Получаем точки из данных 
d = 2*r_p_mean
y_df_pr_V = [0]
for i in range(len(V_p)-1):
    y_df_pr_V.append(V_p[i+1]/(d[i]-d[i+1]))
y_df_pr_log_V = [0]
for i in range(len(V_p)-1):
    y_df_pr_log_V.append(V_p[i+1]/((np.log10(d[i])-np.log10(d[i+1]))))

#Искусственное увеличение точек с помощью аппроксимации функции распределения
sV_p = np.array(sV_p)
idx_sort = np.argsort(d)
sort_d_mean = d[idx_sort]
sort_sV_p = sV_p[idx_sort]
f_V = interpolate.Akima1DInterpolator(sort_d_mean, sort_sV_p)
x_V = np.linspace(sort_d_mean[0], sort_d_mean[-1], 500)
df_V = f_V.derivative()
y_df_V = -df_V(x_V)
y_df_V = np.clip(y_df_V, 0, max(y_df_pr_V))
y_df_log_V = y_df_V * x_V * np.log(10)

#Интерполяции точек данных
y_df_pr_V = np.array(y_df_pr_V)
spline_V = interpolate.Akima1DInterpolator(sort_d_mean, y_df_pr_V[idx_sort])
y_spline_V = spline_V(x_V)
y_df_pr_log_V = np.array(y_df_pr_log_V)
spline_log_V = interpolate.Akima1DInterpolator(sort_d_mean, y_df_pr_log_V[idx_sort])
y_spline_log_V = spline_log_V(x_V)


In [ ]:
#Выходные данные
df = pd.DataFrame()
df2 = pd.DataFrame()
df ['Давление, P/P0'] = data_PP0
df ['Объем, cm3/g'] = data_V
df ['Радиус Кельвина'] = r_k
df ['Толщина монослоя'] = t
df ['Радиус пор'] = r_p
df2 ['Средний радиус Кельвина'] = r_k_mean
df2 ['Средний радиус пор'] = r_p_mean
df2 ['Средний диаметр пор'] = d
df2 ['Изменение толщины монослоя'] = delta_t
df2 ['Изменение адсорбируемого объема (газ)'] = delta_V
df2 ['Изменение адсорбируемого объема (жидкость)'] = delta_V_f
df2 ['S*t'] = s_t
df2 ['Изменение объема пор'] = V_p
df2 ['Изменение площади пор'] = S_p
df2 ['Суммарный объем пор'] = sV_p
df2 ['Суммарная площадь пор'] = sS_p
result = pd.concat([df, df2], axis=1)
print(result)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize = (16, 10))

#ax[0, 0].plot(x, y_df, c = 'r', label = 'Искусственное увеличение точек')
ax[0, 0].scatter(d, y_df_pr, c ='g', marker = 's', label = 'Экспериментальные данные')
ax[0, 0].plot(x, y_spline, c = 'g', label = 'Интерполяция экспериментальных данных')
ax[0, 0].grid(True)
ax[0, 0].set_xlabel('Диаметр пор, Å')
ax[0, 0].set_ylabel('dS/dD')
ax[0, 0].legend(loc = 'upper left')
ax[0, 0].set_xlim(0, 300)

#ax[0, 1].plot(x, y_df_log, c = 'r', label = 'Искусственное увеличение точек')
ax[0, 1].scatter(d, y_df_pr_log, c ='g', marker = 's', label = 'Экспериментальные данные')
ax[0, 1].plot(x, y_spline_log, c = 'g', label = 'Интерполяция экспериментальных данных')
ax[0, 1].grid(True)
ax[0, 1].set_xlabel('Диаметр пор, Å')
ax[0, 1].set_ylabel('dS/d(lg(D))')
ax[0, 1].legend(loc = 'upper left')
ax[0, 1].set_xlim(0, 300)

#ax[1, 0].plot(x_V, y_df_V, c = 'orange', label = 'Искусственное увеличение точек')
ax[1, 0].scatter(d, y_df_pr_V, c ='b', marker = 's', label = 'Экспериментальные данные')
ax[1, 0].plot(x_V, y_spline_V, c = 'b', label = 'Интерполяция экспериментальных данных')
ax[1, 0].grid(True)
ax[1, 0].set_xlabel('Диаметр пор, Å')
ax[1, 0].set_ylabel('dV/dD')
ax[1, 0].legend(loc = 'upper left')
ax[1, 0].set_xlim(0, 300)

#ax[1, 1].plot(x_V, y_df_log_V, c = 'orange', label = 'Искусственное увеличение точек')
ax[1, 1].scatter(d, y_df_pr_log_V, c ='b', marker = 's', label = 'Экспериментальные данные')
ax[1, 1].plot(x_V, y_spline_log_V, c = 'b', label = 'Интерполяция экспериментальных данных')
ax[1, 1].grid(True)
ax[1, 1].set_xlabel('Диаметр пор, Å')
ax[1, 1].set_ylabel('dV/d(lg(D))')
ax[1, 1].legend(loc = 'upper left')
ax[1, 1].set_xlim(0, 300)

plt.show()
plt.tight_layout()